# Complementing Regular Expressions

In the lecture notes we have shown that the complement $\Sigma^* \backslash L$ of a regular language $L$ over the
alphabet $\Sigma$ is again a regular language.  The proof converts a regular expression $r$ with $L = L(r)$ into an
<span style="font-variant:small-caps;">Nfa</span> $F$, which is then converted into the complete <span style="font-variant:small-caps;">Dfa</span> $\mathit{det}(F)$.  Swapping the accepting and the
non-accepting states of this <span style="font-variant:small-caps;">Dfa</span> yields a <span style="font-variant:small-caps;">Dfa</span> that accepts the complement $\Sigma^* \backslash L(r)$.
Therefore, there is a function
$$ \texttt{complementRegExp}: \textsf{RegExp} \rightarrow \textsf{RegExp} $$
such that
$$ L\bigl(\texttt{complementRegExp}(r)\bigr) = \Sigma^* \backslash L(r) $$
holds for every regular expression $r$.  In this notebook we implement this function.  As the complement is taken
with respect to $\Sigma^*$, the function also needs the alphabet $\Sigma$ as an argument:  For example, the
complement of $L(\texttt{a}^*)$ is the empty language if $\Sigma = \{\texttt{a}\}$, but it is the set of all
strings that contain a $\texttt{b}$ if $\Sigma = \{\texttt{a}, \texttt{b}\}$.  Given a regular expression $r$
and an alphabet $\Sigma$, a regular expression for the complement of $L(r)$ is computed as follows:
1. $r$ is converted into an <span style="font-variant:small-caps;">Nfa</span>, which is then converted into a complete <span style="font-variant:small-caps;">Dfa</span> $F$.
2. The <span style="font-variant:small-caps;">Dfa</span> $F$ is minimized.  This step is not needed for the construction, but it keeps the <span style="font-variant:small-caps;">Dfa</span> small.
3. We compute the complement automaton $\overline{F}$ by swapping the accepting and the non-accepting states of $F$.
4. The complement automaton is converted into a regular expression.

We will see that the resulting regular expression is much larger than necessary, even for very simple regular
expressions $r$.

The notebook `09-Equivalence.ipynb` provides the function `regexp2DFA`, which converts a regular expression into a <span style="font-variant:small-caps;">Dfa</span>, and the functions `fsm_complement` and `is_empty`, which we use to check our results.

In [ ]:
%run 09-Equivalence.ipynb

The notebook `07-Minimize.ipynb` provides the function `minimize`, which minimizes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
%run 07-Minimize.ipynb

The notebook `05-DFA-2-RegExp.ipynb` provides the function `dfa_2_regexp`, which converts a <span style="font-variant:small-caps;">Dfa</span> into a regular expression.

In [ ]:
%run 05-DFA-2-RegExp.ipynb

The notebook `Simplify.ipynb` provides the function `simplify`, which simplifies a regular expression, and the function `to_str`, which turns a regular expression into a string that contains only the parentheses that are needed.  The notebook `RegExp-Parser.ipynb` provides the function `parse`, which turns a string into a regular expression.

In [ ]:
%run Simplify.ipynb
%run RegExp-Parser.ipynb

In [ ]:
from typing import TypeVar, Literal
import time

**Note:** The type checker *basedpyright* analyzes every notebook on its own.  Since it cannot follow the
magic command `%run`, the next cell declares the names that are defined in the notebooks loaded above.
These declarations are never executed, since `TYPE_CHECKING` is `False` at run time.  The script
`check_declarations.py` in the directory `Python` checks that they agree with the definitions.

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    type BinaryOp  = Literal['⋅', '+']
    type UnaryOp   = Literal['*']
    Char = str
    type RegExp = int | Char | tuple[RegExp, UnaryOp] | tuple[RegExp, BinaryOp, RegExp]
    State = TypeVar('State')
    StatePair = tuple[State, State]
    TransRel1 = dict[tuple[State, Char], State]
    TransRel2 = dict[tuple[StatePair, Char], StatePair]
    DFA1      = tuple[set[State], set[Char], TransRel1, State, set[State]]
    DFA2      = tuple[set[StatePair], set[Char], TransRel2, StatePair, set[StatePair]]
    TransRel = dict[tuple[State, Char], State]
    DFA      = tuple[set[State], set[Char], TransRel, State, set[State]]
    SetState    = frozenset[State]
    TransRelSet = dict[tuple[SetState, Char], SetState]
    Min_DFA     = tuple[set[SetState], set[Char], TransRelSet, SetState, set[SetState]]
    def regexp2DFA(r: RegExp, Σ: set[Char]) -> DFA1: ...
    def fsm_complement(F1: DFA1, F2: DFA1) -> DFA2: ...
    def is_empty(F: DFA2) -> bool: ...
    def minimize(F: DFA) -> Min_DFA: ...
    def dfa_2_regexp(F: DFA) -> RegExp: ...
    def simplify(r: RegExp) -> RegExp: ...
    def to_str(r: RegExp) -> str: ...
    def parse(s: str) -> RegExp: ...

Given a complete <span style="font-variant:small-caps;">Dfa</span>
$$ F = \langle Q, \Sigma, \delta, q_0, A \rangle, $$
the *complement automaton* $\overline{F}$ is defined as
$$ \overline{F} := \langle Q, \Sigma, \delta, q_0, Q \backslash A \rangle, $$
i.e. the accepting states of $\overline{F}$ are the states of $F$ that are not accepting.  As $F$ is complete,
reading a string $w$ takes $F$ into the state $\delta^*(q_0, w)$, and we have
$$ w \in L\bigl(\overline{F}\bigr) \;\Leftrightarrow\; \delta^*(q_0, w) \in Q \backslash A
   \;\Leftrightarrow\; w \not\in L(F). $$
Hence $\overline{F}$ accepts the language $\Sigma^* \backslash L(F)$.  The construction requires that $F$ is
complete:  If $F$ got stuck on a string $w$ because $\delta$ is undefined for some state and character, then
neither $F$ nor $\overline{F}$ would accept $w$.  The <span style="font-variant:small-caps;">Dfa</span>s computed by the function `nfa2dfa` are complete:
If a character leads to no state of the <span style="font-variant:small-caps;">Nfa</span>, the corresponding transition leads to the empty set, which is then a
state of the <span style="font-variant:small-caps;">Dfa</span>.  The function `minimize` keeps the <span style="font-variant:small-caps;">Dfa</span>s complete.

**Function `complement_automaton(F)`**
- *Input:* `F` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* The complement automaton $\overline{F}$, which accepts the language $\Sigma^* \backslash L(F)$.

In [ ]:
def complement_automaton(F: DFA) -> DFA:
    States, Σ, 𝛿, q0, A = F
    return States, Σ, 𝛿, q0, States - A

**Function `complement_dfa(r, Σ)`**
- *Input:* `r` is a regular expression and `Σ` is the alphabet.
- *Output:* The complement automaton of the minimized <span style="font-variant:small-caps;">Dfa</span> for `r`.  It accepts the language
  $\Sigma^* \backslash L(r)$.

In [ ]:
def complement_dfa(r: RegExp, Σ: set[Char]) -> DFA:
    F = minimize(regexp2DFA(r, Σ))
    return complement_automaton(F)

**Function `complementRegExp(r, Σ)`**
- *Input:* `r` is a regular expression and `Σ` is the alphabet.
- *Output:* A regular expression that describes the language $\Sigma^* \backslash L(r)$.  It is not simplified.

In [ ]:
def complementRegExp(r: RegExp, Σ: set[Char]) -> RegExp:
    return dfa_2_regexp(complement_dfa(r, Σ))

To measure the size of a regular expression we count the nodes of its syntax tree:  Every character, every
occurrence of $\varepsilon$ and $\emptyset$, and every operator counts as one node.

**Function `size(r)`**
- *Input:* `r` is a regular expression, represented as a nested tuple.
- *Output:* The number of nodes of the syntax tree of `r`.

In [ ]:
def size(r: RegExp) -> int:
    match r:
        case (r1, '*'):
            return 1 + size(r1)
        case (r1, _, r2):
            return 1 + size(r1) + size(r2)
        case _:
            return 1

The function `accepts_same_language` checks whether the <span style="font-variant:small-caps;">Dfa</span> computed from a regular expression `r` accepts
the same language as the <span style="font-variant:small-caps;">Dfa</span> `F`.  We use it to check that the regular expressions computed by
`complementRegExp` are correct.

**Function `accepts_same_language(r, F, Σ)`**
- *Input:* `r` is a regular expression, `F` is a complete <span style="font-variant:small-caps;">Dfa</span>, and `Σ` is its alphabet.
- *Output:* `True` if $L(r) = L(F)$, `False` otherwise.

In [ ]:
def accepts_same_language(r: RegExp, F: DFA, Σ: set[Char]) -> bool:
    D = regexp2DFA(r, Σ)
    return is_empty(fsm_complement(D, F)) and is_empty(fsm_complement(F, D))

## A First Example

The regular expression $r$ describes the strings over the alphabet $\{\texttt{a}, \texttt{b}\}$ that contain the
substring $\texttt{ab}$:
$$ r = (\texttt{a} + \texttt{b})^* \cdot \texttt{a} \cdot \texttt{b} \cdot (\texttt{a} + \texttt{b})^*. $$

In [ ]:
Σ = { 'a', 'b' }
r = parse('(a+b)*ab(a+b)*')
size(r)

The minimized <span style="font-variant:small-caps;">Dfa</span> for $r$ has 3 states and so has its complement automaton.

In [ ]:
F = complement_dfa(r, Σ)
len(F[0])

In [ ]:
c = complementRegExp(r, Σ)
size(c)

Although $r$ has only 13 nodes, the regular expression for its complement has 415 nodes.
Let us check that it is correct.

In [ ]:
accepts_same_language(c, F, Σ)

After simplification, the regular expression is much smaller:

In [ ]:
s = simplify(c)
size(s)

In [ ]:
print(to_str(s))

In [ ]:
accepts_same_language(s, F, Σ)

This is exactly the regular expression that a human would write:  A string does not contain the substring
$\texttt{ab}$ iff no $\texttt{b}$ follows an $\texttt{a}$, i.e. iff all of its $\texttt{b}$s precede all of its
$\texttt{a}$s.  This leads to the regular expression
$$ \texttt{b}^* \cdot \texttt{a}^*. $$

In [ ]:
s == parse('b*a*')

## Growth of the Regular Expressions

The function `dfa_2_regexp` calls the function `rpq`, which calls itself four times for every state of the
<span style="font-variant:small-caps;">Dfa</span>.  Therefore, the size of the regular expression computed for a <span style="font-variant:small-caps;">Dfa</span> with $n$ states grows like $4^n$.
As the complement automaton has as many states as the minimized <span style="font-variant:small-caps;">Dfa</span> for $r$, the size of the complement grows
exponentially in this number.  The following table shows this for the languages of strings in which the number of
`a`s is divisible by $k$.  The complements of these languages consist of the strings in which the number of `a`s is
not divisible by $k$.  For $k = 1$, the language contains all strings, so its complement is empty.

**Function `divisible(c, d, k)`**
- *Input:* `c` and `d` are the two characters of the alphabet and `k` is a positive natural number.
- *Output:* A string that is parsed into a regular expression describing the strings over the alphabet
  $\{c, d\}$ in which the number of occurrences of `c` is divisible by `k`.

In [ ]:
def divisible(c: Char, d: Char, k: int) -> str:
    return '(' + f'{d}*{c}' * k + f')*{d}*'

In [ ]:
print(divisible('a', 'b', 2))
print(divisible('a', 'b', 3))

In [ ]:
print(f'{"k":>2} {"states":>7} {"size(r)":>8} {"size(complement)":>17} {"simplified":>11} {"correct":>8} {"time":>7}')
for k in range(1, 9):
    r     = parse(divisible('a', 'b', k))
    start = time.perf_counter()
    F     = complement_dfa(r, Σ)
    c     = dfa_2_regexp(F)
    s     = simplify(c)
    stop  = time.perf_counter()
    ok    = accepts_same_language(s, F, Σ)
    print(f'{k:>2} {len(F[0]):>7} {size(r):>8} {size(c):>17} {size(s):>11} {ok!s:>8} {stop - start:>6.2f}s')

Although the regular expressions $r$ are tiny, the regular expression for the complement has almost
1&nbsp;400&nbsp;000 nodes when the minimized <span style="font-variant:small-caps;">Dfa</span> has only eight states.  Simplification removes almost all of
these nodes:  For $k = 8$, the simplified regular expression has only 126 nodes.  Nevertheless, we do not compute
the complement of a regular expression in practice, since the regular expression computed by `dfa_2_regexp` grows
exponentially:  If we need the complement of a regular language, we work with the complement automaton instead.